# ORTHRUS-MSTC-PIDS: All-in-One Colab Master Notebook

**Version**: 1.1.0  
**Frozen Tag**: `mstc-pids-c1-c8-exp-v1`  
**Commit**: `fix/c8-preprocess-memory` (local development branch)

---

## 设计原则

1. **单一 Runtime**: 整个实验流程在同一个 Colab Runtime 中完成
2. **幂等安装**: 依赖已存在且版本可用时不强制重新安装
3. **Drive 持久化**: artifacts, checkpoints, metrics, results 写入 Google Drive
4. **安全恢复**: 数据库恢复有明确的前置条件和保护机制

---

## 流程概览

| 阶段 | 描述 | 默认 |
|------|------|------|
| 0. 全局参数 | 项目路径、仓库版本、数据集配置 | - |
| 1. Google Drive | 挂载 Drive 并创建目录 | `True` |
| 2. 冻结代码版本 | Clone 并 checkout 到冻结 tag | `True` |
| 3. GPU/CUDA | 验证 GPU 可用性 | - |
| 4. 依赖安装 | Python/PyG 依赖 | `True` |
| 5. 环境记录 | 保存 pip freeze | `True` |
| 6. 数据检查 | 检查预处理产物 | - |
| 7. PostgreSQL | 安装/启动/恢复数据库 | `False` |
| 8. 预处理 | 执行预处理（需要数据库） | `False` |
| 9. Baseline Smoke | 1-epoch smoke test | `False` |
| 10. 主模型矩阵 | ORTHRUS-ano + MSTC-PIDS Full | `False` |
| 11. 消融实验 | A0~A6 及各专项 | `False` |
| 12. 结果收集 | collect_results + export_tables | `False` |
| 13. 结果展示 | 显示 CSV 表格 | `False` |

---

## 0. 全局参数

**重要**: 修改此区域即可配置整个实验。所有路径使用 Google Drive 约定。

In [ ]:
# ============================================================
# 用户参数：按需修改这一格即可
# ============================================================

from pathlib import Path

# --- 路径配置 ---
PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"

# --- 数据库 Dump 配置 ---
DB_DUMPS = {
    "THEIA_E3": DATA_ROOT / "database_dumps" / "theia_e3.dump",
    "THEIA_E5": DATA_ROOT / "database_dumps" / "theia_e5.dump",
}

# --- 仓库配置（必须固定到冻结 tag）---
REPOSITORY_URL = "https://github.com/fish23611-beep/orthrus.git"
REPOSITORY_REF = "mstc-pids-c1-c8-exp-v1"
EXPECTED_COMMIT = "0a7ab00bb0900df5afd4ecb01359ab10ad37c60e"

# --- 实验配置 ---
DATASET = "THEIA_E3"
SEEDS = [0]
EXPERIMENT_GROUP = "ablation"

# --- 阶段开关 ---
MOUNT_DRIVE = True
CLONE_OR_UPDATE = True
INSTALL_DEPENDENCIES = True
RESTORE_DATABASE = False
FORCE_DATABASE_RESTORE = False
RUN_PREPROCESS = False
FORCE_PREPROCESS = False

# --- C8.1 Preprocessing Substages (bounded-memory) ---
# 可选值: "build_graphs" | "embed_nodes" | "embed_edges" | "build_graphs,embed_nodes,embed_edges"
# 默认运行所有三个阶段。Runtime 崩溃后可单独运行各阶段恢复
PREPROCESS_SUBSTAGES = "build_graphs,embed_nodes,embed_edges"
RUN_BASELINE_SMOKE = False
RUN_MAIN_MATRIX = False
RUN_ABLATIONS = False
RUN_COLLECT_EXPORT = False
RUN_DISPLAY_RESULTS = False
RUN_MANUAL_RESUME = False
RESUME_CONFIG = PROJECT_ROOT / "config/experiments/mstc_full.yml"
CHECKPOINT = Path("")

# --- 派生变量 ---
DATABASE_DUMP = DB_DUMPS.get(DATASET, Path(""))
assert DATASET in {"THEIA_E3", "THEIA_E5"}

print("=" * 60)
print("全局参数已设置")
print("=" * 60)
print(f"PROJECT_ROOT:    {PROJECT_ROOT}")
print(f"DRIVE_ROOT:      {DRIVE_ROOT}")
print(f"DATASET:         {DATASET}")
print(f"REPOSITORY_REF:  {REPOSITORY_REF}")
print(f"EXPECTED_COMMIT: {EXPECTED_COMMIT}")
print("=" * 60)

---

## 1. Google Drive 挂载

In [ ]:
import os

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        print("挂载 Google Drive...")
        drive.mount("/content/drive")
        print("Google Drive 已挂载。")
    except ImportError:
        print("当前不是 Colab 环境；跳过 Google Drive 挂载。")

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "database_dumps").mkdir(parents=True, exist_ok=True)

os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)

print(f"ORTHRUS_ARTIFACT_ROOT = {os.environ['ORTHRUS_ARTIFACT_ROOT']}")

---

## 2. 冻结代码版本

In [ ]:
import subprocess
import sys

if CLONE_OR_UPDATE:
    if (PROJECT_ROOT / ".git").is_dir():
        print(f"检测到现有仓库: {PROJECT_ROOT}")
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "--all", "--tags"], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPOSITORY_REF], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    elif PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f"项目路径已存在但不是 Git 仓库: {PROJECT_ROOT}")
    else:
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--branch", REPOSITORY_REF, "--recurse-submodules", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

    result = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], capture_output=True, text=True, check=True)
    actual_commit = result.stdout.strip()
    if actual_commit != EXPECTED_COMMIT:
        raise RuntimeError(f"Commit 不匹配！期望 {EXPECTED_COMMIT}，实际 {actual_commit}")
    print(f"✓ Commit 验证通过: {actual_commit}")

src_root = str(PROJECT_ROOT / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)

---

## 3. GPU / CUDA 验证

In [ ]:
import shutil
import subprocess
import sys

print("=" * 60)
print("GPU / CUDA 验证")
print("=" * 60)

print(f"Python: {sys.version}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"Torch CUDA: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=True)

subprocess.run(["df", "-h", str(PROJECT_ROOT.parent)], check=True)

GPU_READY = torch.cuda.is_available()
print(f"GPU_READY: {GPU_READY}")
print("=" * 60)

In [ ]:
# ============================================================================
# 资源预检：显示系统资源状态
# ============================================================================
import psutil
import os
import subprocess
import shutil

print("=" * 60)
print("资源预检")
print("=" * 60)

# CPU RAM
ram = psutil.virtual_memory()
print(f"总 RAM:     {ram.total / (1024**3):.1f} GB")
print(f"可用 RAM:   {ram.available / (1024**3):.1f} GB")
print(f"RAM 使用率: {ram.percent}%")
if ram.percent > 80:
    print("⚠️  警告: RAM 使用率较高，可能影响 preprocessing")

# GPU
if shutil.which("nvidia-smi"):
    print("\nGPU 信息:")
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], check=False)
else:
    print("\nGPU: 不可用 (CPU-only 环境)")

# Disk space
for path in ["/content", str(DRIVE_ROOT)]:
    if os.path.exists(path):
        usage = shutil.disk_usage(path)
        used_percent = usage.used / usage.total * 100
        print(f"\n{path} 磁盘: 总计 {usage.total / (1024**3):.1f} GB, "
              f"可用 {usage.free / (1024**3):.1f} GB ({used_percent:.1f}% used)")

print("=" * 60)
print("资源预检完成")
print("=" * 60)


---

## 4. Python / PyG 依赖安装

In [ ]:
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    base_packages = ["scikit-learn", "networkx", "xxhash", "graphviz", "psutil", "matplotlib", "wandb", "chardet", "nltk", "igraph", "cairocffi", "wget", "gensim", "pytz", "pandas", "yacs", "psycopg2-binary", "tqdm", "pyyaml"]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet"] + base_packages, check=True)
    print("基础依赖安装完成")

    torch_base = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    pyg_index = f"https://data.pyg.org/whl/torch-{torch_base}+{cuda_tag}.html"
    print(f"PyG wheel index: {pyg_index}")

    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "torch_geometric"], check=True)
    print("torch_geometric 已安装")

    pyg_packages = ["pyg_lib", "torch_scatter", "torch_sparse"]
    for pkg in pyg_packages:
        result = subprocess.run([sys.executable, "-m", "pip", "show", pkg], capture_output=True)
        if result.returncode != 0:
            print(f"安装 {pkg}...")
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg, "-f", pyg_index], check=True)
        else:
            print(f"{pkg} 已存在")
    print("PyG 核心包安装完成")

---

## 5. Import Smoke Test + 环境记录

In [ ]:
import importlib
import subprocess
import sys

print("=" * 60)
print("Import Smoke Test")
print("=" * 60)

core_modules = ["config", "orthrus", "torch", "torch_geometric", "pandas", "yaml"]
for module_name in core_modules:
    module = importlib.import_module(module_name)
    version = getattr(module, "__version__", "")
    print(f"✓ import {module_name} {version}")

if torch.cuda.is_available():
    x = torch.randn(100, 100, device="cuda")
    y = x @ x.T
    print(f"✓ GPU tensor test passed")
    del x, y
    torch.cuda.empty_cache()

environment_dir = ARTIFACT_ROOT / "environment"
environment_dir.mkdir(parents=True, exist_ok=True)
freeze_path = environment_dir / "pip_freeze.txt"
with freeze_path.open("w", encoding="utf-8") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, text=True, check=True)
print(f"pip freeze 已保存: {freeze_path}")

result = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], capture_output=True, text=True)
git_commit = result.stdout.strip() if result.returncode == 0 else "unknown"
result = subprocess.run(["git", "-C", str(PROJECT_ROOT), "describe", "--tags", "--exact-match", "HEAD"], capture_output=True, text=True)
git_tag = result.stdout.strip() if result.returncode == 0 else "no tag"
print(f"Git commit: {git_commit}")
print(f"Git tag: {git_tag}")
print("=" * 60)

---

## 6. THEIA 数据检查

In [ ]:
from pathlib import Path
import sys

print("=" * 60)
print("THEIA 数据检查")
print("=" * 60)

from config import get_runtime_required_args, get_yml_cfg

PREPROCESS_CONFIG = PROJECT_ROOT / "config/orthrus.yml"
config_args = get_runtime_required_args(args=[DATASET, "--config", str(PREPROCESS_CONFIG), "--artifact-root", str(ARTIFACT_ROOT), "--stages", "preprocess", "--skip-tracing"])
cfg = get_yml_cfg(config_args)

required_paths = {
    "graphs": Path(cfg.graph_construction.build_graphs._graphs_dir),
    "word2vec": Path(cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir),
    "edge_embeddings": Path(cfg.edge_featurization.embed_edges._edge_embeds_dir),
    "metadata": Path(cfg._metadata_dir),
}


# C8.1: 使用 completion markers 进行更精确的状态检测
def check_completion_marker(artifact_dir, marker_name):
    marker_path = artifact_dir / marker_name
    return marker_path.exists()

# 检查 markers
graphs_dir = Path(cfg.graph_construction.build_graphs._graphs_dir)
word2vec_dir = Path(cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir)
edge_embeds_dir = Path(cfg.edge_featurization.embed_edges._edge_embeds_dir)

marker_status = {
    "build_graphs": check_completion_marker(graphs_dir, ".preprocess_build_graphs_complete"),
    "embed_nodes": check_completion_marker(word2vec_dir, ".preprocess_embed_nodes_complete"),
    "embed_edges": check_completion_marker(edge_embeds_dir, ".preprocess_embed_edges_complete"),
}
print("\nC8.1 Completion Markers:")
for stage, complete in marker_status.items():
    print(f"  {stage}: {'✓' if complete else '✗'}")


def visible_entries(path):
    return [item for item in path.rglob("*") if item.is_file()] if path.is_dir() else []

for label, path in required_paths.items():
    count = len(visible_entries(path))
    status = "✓" if count > 0 else "✗"
    print(f"{status} {label}: {path} (文件数={count})")

artifacts_complete = all(len(visible_entries(path)) > 0 for path in required_paths.values())
print(f"\n完整预处理产物: {artifacts_complete}")
print("=" * 60)

---

## 7. PostgreSQL

In [ ]:
import os
import secrets
import subprocess

if not RESTORE_DATABASE:
    print("RESTORE_DATABASE=False，不触碰 PostgreSQL。")
    if artifacts_complete:
        print("已有预处理产物，可直接用于 detection-only。")
else:
    print("=" * 60)
    print("PostgreSQL 设置")
    print("=" * 60)

    if not subprocess.run(["which", "pg_isready"], capture_output=True).returncode == 0:
        print("安装 postgresql...")
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-qq", "-y", "postgresql", "postgresql-contrib"], check=True)

    print("启动 PostgreSQL...")
    subprocess.run(["service", "postgresql", "start"], check=True)

    import time
    for _ in range(30):
        if subprocess.run(["pg_isready", "-h", "localhost", "-p", "5432"], capture_output=True).returncode == 0:
            break
        time.sleep(1)

    os.environ["ORTHRUS_DB_HOST"] = "localhost"
    os.environ["ORTHRUS_DB_PORT"] = "5432"
    os.environ["ORTHRUS_DB_USER"] = "postgres"
    os.environ["ORTHRUS_DB_PASSWORD"] = secrets.token_urlsafe(16)

    pg_env = os.environ.copy()
    pg_env["PGPASSWORD"] = os.environ["ORTHRUS_DB_PASSWORD"]
    subprocess.run(["psql", "-h", "localhost", "-p", "5432", "-U", "postgres", "-c", f"ALTER USER postgres PASSWORD '{os.environ['ORTHRUS_DB_PASSWORD']}';"], env=pg_env, check=True)

    db_name = cfg.dataset.database_all_file if cfg.graph_construction.build_graphs.use_all_files else cfg.dataset.database
    print(f"数据库名称: {db_name}")

    if not DATABASE_DUMP.is_file():
        raise FileNotFoundError(f"数据库 dump 文件不存在: {DATABASE_DUMP}")

    result = subprocess.run(["psql", "-h", "localhost", "-p", "5432", "-U", "postgres", "-lqt", "-c", f"SELECT 1 FROM pg_database WHERE datname='{db_name}';"], env=pg_env, capture_output=True, text=True)
    db_exists = db_name in result.stdout

    if db_exists:
        if FORCE_DATABASE_RESTORE:
            print(f"数据库 {db_name} 已存在，删除并重建...")
            subprocess.run(["psql", "-h", "localhost", "-p", "5432", "-U", "postgres", "-c", f"DROP DATABASE IF EXISTS {db_name};"], env=pg_env, check=True)
        else:
            print(f"数据库 {db_name} 已存在，跳过恢复。")
    else:
        subprocess.run(["psql", "-h", "localhost", "-p", "5432", "-U", "postgres", "-c", f"CREATE DATABASE {db_name};"], env=pg_env, check=True)

    if not db_exists or FORCE_DATABASE_RESTORE:
        print(f"恢复数据库 {db_name}...")
        subprocess.run(["pg_restore", "--no-owner", "-h", "localhost", "-p", "5432", "-U", "postgres", "-d", db_name, str(DATABASE_DUMP)], env=pg_env, check=True)
        print(f"数据库 {db_name} 恢复完成")

    print("=" * 60)

---

## 8. Preprocessing

In [ ]:
import subprocess
import sys
from pathlib import Path

if not RUN_PREPROCESS:
    print("RUN_PREPROCESS=False，跳过预处理。")
    if not artifacts_complete:
        print("⚠️  警告: 预处理产物不完整")
elif artifacts_complete and not FORCE_PREPROCESS:
    print("检测到完整产物，跳过预处理。")
else:
    print("=" * 60)
    print("开始预处理")
    print("=" * 60)

    graphs_dir = Path(cfg.graph_construction.build_graphs._graphs_dir)
    word2vec_dir = Path(cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir)
    edge_embeds_dir = Path(cfg.edge_featurization.embed_edges._edge_embeds_dir)
    markers = {
        "build_graphs": graphs_dir / ".preprocess_build_graphs_complete",
        "embed_nodes": word2vec_dir / ".preprocess_embed_nodes_complete",
        "embed_edges": edge_embeds_dir / ".preprocess_embed_edges_complete",
    }
    print("Preprocessing stage status:")
    for stage, marker in markers.items():
        print(f"  {stage}: {'✓' if marker.exists() else '✗'}")

    command = [
        sys.executable, str(PROJECT_ROOT / "src/orthrus.py"), DATASET,
        "--config", str(PREPROCESS_CONFIG), "--stages", "preprocess",
        "--preprocess-substages", PREPROCESS_SUBSTAGES, "--skip-tracing",
        "--artifact-root", str(ARTIFACT_ROOT),
    ]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print("预处理完成")
    print("=" * 60)


---

## 9. Baseline Smoke Test

In [ ]:
import subprocess
import sys
import yaml
from pathlib import Path

if not RUN_BASELINE_SMOKE:
    print("RUN_BASELINE_SMOKE=False，跳过 baseline smoke test。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("Baseline Smoke Test")
    print("=" * 60)

    BASE_CONFIG = PROJECT_ROOT / "config/experiments/baseline.yml"
    SMOKE_CONFIG = ARTIFACT_ROOT / "environment/smoke_baseline_1epoch.yml"

    smoke = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
    smoke.setdefault("pipeline", {})["run_tracing"] = False
    smoke.setdefault("detection", {}).setdefault("gnn_training", {})["num_epochs"] = 1
    SMOKE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    SMOKE_CONFIG.write_text(yaml.safe_dump(smoke, sort_keys=False), encoding="utf-8")

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_experiment.py"), "--dataset", DATASET, "--config", str(SMOKE_CONFIG), "--seed", str(SEEDS[0]), "--artifact-root", str(ARTIFACT_ROOT), "--stages", "train,test,evaluate"]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print("Baseline smoke 通过")
    print("=" * 60)

---

## 10. 主模型矩阵

In [ ]:
import subprocess
import sys

if not RUN_MAIN_MATRIX:
    print("RUN_MAIN_MATRIX=False，跳过主模型矩阵。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("主模型矩阵")
    print("=" * 60)

    MAIN_CONFIGS = [PROJECT_ROOT / "config/experiments/baseline.yml", PROJECT_ROOT / "config/experiments/mstc_full.yml"]

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_matrix.py"), "--datasets", DATASET, "--configs", ",".join(map(str, MAIN_CONFIGS)), "--seeds", ",".join(map(str, SEEDS)), "--artifact-root", str(ARTIFACT_ROOT)]
    result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)

    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    print("矩阵完成")
    print("=" * 60)

---

## 11. 消融与专项实验

In [ ]:
import subprocess
import sys
from pathlib import Path

if not RUN_ABLATIONS:
    print("RUN_ABLATIONS=False，跳过消融实验。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print(f"消融与专项实验: {EXPERIMENT_GROUP}")
    print("=" * 60)

    GROUPS = {
        "ablation": ["baseline.yml", "ablation_no_multiscale.yml", "ablation_no_gate.yml", "ablation_no_time.yml", "ablation_no_calibration.yml", "ablation_no_topk.yml", "mstc_full.yml"],
        "multiscale": ["multiscale_recent20.yml", "multiscale_recent24.yml", "multiscale_single_window.yml", "multiscale_equal.yml", "multiscale_gate.yml"],
        "time": ["time_type_only.yml", "time_time_only.yml", "time_joint.yml"],
        "calibration": ["calibration_max.yml", "calibration_quantile.yml", "calibration_kmeans.yml", "calibration_global_p.yml", "calibration_relation.yml", "calibration_hierarchical.yml"],
        "backbone": ["backbone_graphtransformer.yml", "backbone_graphsage_baseline.yml", "backbone_graphsage.yml", "backbone_mlp.yml"],
        "dataset_view": ["host_only.yml", "host_network_structure.yml", "host_network_full.yml"],
        "efficiency": ["baseline.yml", "efficiency_multiscale.yml", "efficiency_multiscale_time.yml", "mstc_full.yml"],
    }

    if EXPERIMENT_GROUP not in GROUPS:
        raise ValueError(f"未知组 {EXPERIMENT_GROUP!r}；可选 {sorted(GROUPS.keys())}")

    CONFIGS = [PROJECT_ROOT / "config/experiments" / name for name in GROUPS[EXPERIMENT_GROUP]]

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_matrix.py"), "--datasets", DATASET, "--configs", ",".join(map(str, CONFIGS)), "--seeds", ",".join(map(str, SEEDS)), "--artifact-root", str(ARTIFACT_ROOT)]
    result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)

    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    print("消融实验完成")
    print("=" * 60)

---

## 12. Checkpoint Resume

In [ ]:
import subprocess
import sys

if not RUN_MANUAL_RESUME:
    print("RUN_MANUAL_RESUME=False，未加载任何 checkpoint。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("Checkpoint Resume")
    print("=" * 60)

    if not CHECKPOINT or not Path(CHECKPOINT).exists():
        raise FileNotFoundError(f"CHECKPOINT 不存在: {CHECKPOINT}")

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_experiment.py"), "--dataset", DATASET, "--config", str(RESUME_CONFIG), "--seed", str(SEEDS[0]), "--artifact-root", str(ARTIFACT_ROOT), "--stages", "train,test,evaluate", "--checkpoint", str(CHECKPOINT)]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print("Resume 完成")
    print("=" * 60)

---

## 13. 结果收集与导出

In [ ]:
import subprocess
import sys

if not RUN_COLLECT_EXPORT:
    print("RUN_COLLECT_EXPORT=False，跳过结果收集与导出。")
else:
    print("=" * 60)
    print("结果收集与导出")
    print("=" * 60)

    collect_command = [sys.executable, str(PROJECT_ROOT / "src/experiments/collect_results.py"), "--artifact-root", str(ARTIFACT_ROOT)]
    subprocess.run(collect_command, cwd=PROJECT_ROOT, check=True)

    export_command = [sys.executable, str(PROJECT_ROOT / "src/experiments/export_tables.py"), "--artifact-root", str(ARTIFACT_ROOT)]
    subprocess.run(export_command, cwd=PROJECT_ROOT, check=True)

    print("结果收集与导出完成")
    print("=" * 60)

---

## 14. 结果展示

In [ ]:
from pathlib import Path
import pandas as pd

if not RUN_DISPLAY_RESULTS:
    print("RUN_DISPLAY_RESULTS=False，跳过结果展示。")
else:
    print("=" * 60)
    print("结果展示")
    print("=" * 60)

    RESULTS_ROOT = ARTIFACT_ROOT / "results"
    table_names = ["all_runs.csv", "main_results.csv", "ablation_results.csv", "calibration_results.csv", "efficiency_results.csv"]

    for name in table_names:
        path = RESULTS_ROOT / name
        if not path.is_file():
            print(f"警告: {name} 不存在")
            continue
        df = pd.read_csv(path)
        print(f"\n--- {name} ({len(df)} rows) ---")
        display(df)

    print("=" * 60)